In [1]:
# ============================================================
# BLOQUE 1: Importacion de librerias
# ============================================================

import pandas as pd
import numpy as np
import joblib
import anthropic
import os
from dotenv import load_dotenv

# Cargar variables de entorno desde .env
load_dotenv('../.env')

# Inicializar cliente de Anthropic
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

print("Librerias importadas correctamente")
print(f"   anthropic version: {anthropic.__version__}")
print("   Cliente Anthropic inicializado correctamente")

Librerias importadas correctamente
   anthropic version: 0.89.0
   Cliente Anthropic inicializado correctamente


In [2]:
# ============================================================
# BLOQUE 2: Carga de datos y resultados SHAP
# ============================================================

# Datos procesados
X_test  = pd.read_csv('../data/processed/X_test.csv')
y_test  = pd.read_csv('../data/processed/y_test.csv').squeeze()

# Modelos entrenados
xgb_model = joblib.load('../models/xgboost.pkl')

# Resultados de modelos
resultados = joblib.load('../models/resultados_modelos.pkl')
umbral_opt_xgb = resultados['umbral_opt_xgb']

# Datos SHAP
shap_data       = joblib.load('../models/shap_data.pkl')
shap_values_xgb = shap_data['shap_values_xgb']
X_test_muestra  = shap_data['X_test_muestra']
y_test_muestra  = shap_data['y_test_muestra']
muestra_idx     = shap_data['muestra_idx']

print("Datos cargados correctamente")
print(f"X_test         : {X_test.shape}")
print(f"shap_values_xgb: {shap_values_xgb.shape}")
print(f"Umbral XGBoost : {umbral_opt_xgb:.4f}")
print(f"Casos positivos en muestra: {(y_test_muestra == 1).sum()}")

Datos cargados correctamente
X_test         : (16000, 170)
shap_values_xgb: (2000, 170)
Umbral XGBoost : 0.0100
Casos positivos en muestra: 39


In [10]:
# ============================================================
# BLOQUE 3: Funcion de construccion de prompt y diagnostico
# ============================================================

def obtener_top_shap(shap_vals, feature_names, top_n=5):
    indices = np.argsort(np.abs(shap_vals))[::-1][:top_n]
    resultado = []
    for i in indices:
        resultado.append({
            'sensor'    : feature_names[i],
            'valor_shap': round(float(shap_vals[i]), 4),
            'direccion' : 'aumenta riesgo de falla' if shap_vals[i] > 0 else 'reduce riesgo de falla'
        })
    return resultado


def generar_diagnostico(caso_idx, perfil_usuario='operador'):
    prob       = xgb_model.predict_proba(X_test_muestra.iloc[[caso_idx]])[0][1]
    prediccion = 'FALLA APS' if prob >= umbral_opt_xgb else 'NORMAL'
    real       = 'FALLA APS' if y_test_muestra.iloc[caso_idx] == 1 else 'NORMAL'

    top_shap = obtener_top_shap(
        shap_values_xgb[caso_idx],
        X_test_muestra.columns.tolist(),
        top_n=5
    )

    shap_texto = '\n'.join([
        f"   - {s['sensor']}: SHAP={s['valor_shap']} -> {s['direccion']}"
        for s in top_shap
    ])

    prompt = f"""Eres un asistente experto en mantenimiento predictivo industrial.
Se te proporciona el resultado de un modelo de machine learning que analiza datos de sensores
de camiones Scania para detectar fallas en el sistema de presion de aire (APS).

RESULTADO DEL MODELO:
- Prediccion      : {prediccion}
- Probabilidad    : {prob:.4f}
- Umbral utilizado: {umbral_opt_xgb:.4f}
- Valor real      : {real}

VARIABLES MAS INFLUYENTES (SHAP):
{shap_texto}

PERFIL DEL USUARIO: {perfil_usuario}

INSTRUCCIONES:
1. Redacta un diagnostico claro y util para un {perfil_usuario}.
2. Explica que significa la prediccion y que sensores influyeron mas.
3. Sugiere una accion concreta de mantenimiento.
4. Usa unicamente la informacion proporcionada, no inventes datos adicionales.
5. El diagnostico debe tener entre 100 y 150 palabras.
6. No uses formato markdown, solo texto plano."""

    mensaje = client.messages.create(
        model      = "claude-haiku-4-5-20251001",
        max_tokens = 500,
        messages   = [{"role": "user", "content": prompt}]
    )

    return {
        'caso_idx'    : caso_idx,
        'prediccion'  : prediccion,
        'probabilidad': prob,
        'real'        : real,
        'perfil'      : perfil_usuario,
        'diagnostico' : mensaje.content[0].text,
        'top_shap'    : top_shap
    }

print("Funciones definidas correctamente")


Funciones definidas correctamente


In [11]:
# ============================================================
# BLOQUE 4: Generacion de diagnosticos para casos positivos
# ============================================================

# Indices de casos positivos en la muestra
indices_positivos = y_test_muestra[y_test_muestra == 1].index.tolist()
print(f"Casos positivos disponibles: {len(indices_positivos)}")

# Se generan diagnosticos para los primeros 3 casos positivos
# con los tres perfiles de usuario definidos en la metodologia
casos_a_analizar = indices_positivos[:3]
perfiles = ['operador', 'ingeniero', 'gerente']

diagnosticos = []

for caso_idx in casos_a_analizar:
    print(f"\nGenerando diagnosticos para caso {caso_idx}...")
    for perfil in perfiles:
        resultado = generar_diagnostico(caso_idx, perfil)
        diagnosticos.append(resultado)
        print(f"   Perfil '{perfil}' completado.")

print(f"\nTotal de diagnosticos generados: {len(diagnosticos)}")

Casos positivos disponibles: 39

Generando diagnosticos para caso 37...
   Perfil 'operador' completado.
   Perfil 'ingeniero' completado.
   Perfil 'gerente' completado.

Generando diagnosticos para caso 44...
   Perfil 'operador' completado.
   Perfil 'ingeniero' completado.
   Perfil 'gerente' completado.

Generando diagnosticos para caso 61...
   Perfil 'operador' completado.
   Perfil 'ingeniero' completado.
   Perfil 'gerente' completado.

Total de diagnosticos generados: 9


In [13]:
# ============================================================
# BLOQUE 5: Visualizacion de diagnosticos generados
# ============================================================

for d in diagnosticos:
    print("=" * 70)
    print(f"CASO {d['caso_idx']} | Perfil: {d['perfil'].upper()}")
    print(f"Prediccion: {d['prediccion']} | Probabilidad: {d['probabilidad']:.4f} | Real: {d['real']}")
    print("-" * 70)
    print("Top 5 sensores SHAP:")
    for s in d['top_shap']:
        print(f"   {s['sensor']:10} SHAP={s['valor_shap']:>8.4f}  ->  {s['direccion']}")
    print("-" * 70)
    print("Diagnostico:")
    print(d['diagnostico'])
    print()


CASO 37 | Perfil: OPERADOR
Prediccion: FALLA APS | Probabilidad: 0.9995 | Real: FALLA APS
----------------------------------------------------------------------
Top 5 sensores SHAP:
   ck_000     SHAP=  1.5106  ->  aumenta riesgo de falla
   bk_000     SHAP=  1.1834  ->  aumenta riesgo de falla
   aa_000     SHAP=  1.1348  ->  aumenta riesgo de falla
   ai_000     SHAP=  0.8048  ->  aumenta riesgo de falla
   bl_000     SHAP=  0.8007  ->  aumenta riesgo de falla
----------------------------------------------------------------------
Diagnostico:
DIAGNOSTICO DE FALLA APS - CAMION SCANIA

El sistema de presion de aire (APS) presenta una falla confirmada con certeza muy alta (99.95%). El modelo detecto anomalias criticas en cinco sensores principales.

Los sensores ck_000, bk_000 y aa_000 mostraron las lecturas mas anormales y ejercieron mayor influencia en la deteccion de falla. Los sensores ai_000 y bl_000 tambien contribuyeron significativamente.

ACCION RECOMENDADA: Detener la operacio

In [14]:
# ============================================================
# BLOQUE 6: Guardado de diagnosticos
# ============================================================

import json
import os

os.makedirs('../outputs', exist_ok=True)

# Preparar datos para guardar (convertir numpy a tipos nativos)
diagnosticos_export = []
for d in diagnosticos:
    diagnosticos_export.append({
        'caso_idx'    : int(d['caso_idx']),
        'prediccion'  : d['prediccion'],
        'probabilidad': float(d['probabilidad']),
        'real'        : d['real'],
        'perfil'      : d['perfil'],
        'diagnostico' : d['diagnostico'],
        'top_shap'    : [
            {
                'sensor'    : s['sensor'],
                'valor_shap': float(s['valor_shap']),
                'direccion' : s['direccion']
            }
            for s in d['top_shap']
        ]
    })

# Guardar como JSON
with open('../outputs/09_diagnosticos_claude.json', 'w', encoding='utf-8') as f:
    json.dump(diagnosticos_export, f, ensure_ascii=False, indent=2)

print("Diagnosticos guardados en outputs/09_diagnosticos_claude.json")
print(f"Total de diagnosticos guardados: {len(diagnosticos_export)}")

Diagnosticos guardados en outputs/09_diagnosticos_claude.json
Total de diagnosticos guardados: 9
